# Guia do projeto — cinco minutos

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
resumo=json.loads((REPORT/'resumo.json').read_text(encoding='utf-8'))
print(json.dumps(resumo,ensure_ascii=False,indent=2))


{
  "resolucao": "5min",
  "horizonte_minutos": 5,
  "linhas_australia_brutas": 1256431,
  "duplicatas_australia": 0,
  "australia": {
    "grade": 1270529,
    "alvos_medidos": 1210930,
    "inputs_estimados": 39596
  },
  "brasil": {
    "grade": 105120,
    "alvos_medidos": 90023,
    "inputs_estimados": 12627,
    "estacao_temperatura_presente": 33383
  },
  "alvos_imputados": 0,
  "meteo_brasil": "ERA5 horária carregada para frente até 55 minutos; somente auxiliar, excluída do contrato principal",
  "reconstrucao": "Proxy causal e ffill de até três passos nos inputs; sem interpolação com ponto posterior. Alvos exigem medição original.",
  "fontes_sha256": {
    "australia": "adebe983a30f58c5900247308790ed0659de9fdff2a4fd59f6e464147e3c2d14",
    "brasil": "598ede357558a5655894945c9d3aed845dd3fb5ed9e61e25d1f70cbec8611314",
    "estacao": "c51aaf7f7486c8af703bbaaa01c99105b1c5cbd3d73a9543ee93494fae3158be"
  }
}


In [3]:
exibir(pd.read_csv(PASTA/'resultados/particoes.csv'))
print('Inputs:',CONTRATO['variaveis_por_passo'])
print('Históricos:',[l*5 for l in CONFIG['lags']],'minutos; horizonte: 5 minutos')
print(CONTRATO['normalizacao'])


Inputs: ['Active_Power', 'hora_sin', 'hora_cos', 'dia_ano_sin', 'dia_ano_cos']
Históricos: [30, 60] minutos; horizonte: 5 minutos
Potência AC em p.u.; escalas do treino australiano diurno compartilhadas no Brasil.


,dominio,lag,historico_min,particao,janelas_antes_coorte,janelas,primeiro_alvo,ultimo_alvo
0,australia,6,30,treino,488964,40747,2013-07-25T17:05:00,2022-12-31T18:25:00
1,australia,12,60,treino,488448,40704,2013-07-25T17:35:00,2022-12-31T18:25:00
2,australia,6,30,validacao,46840,46840,2023-01-01T05:55:00,2023-12-31T19:20:00
3,australia,12,60,validacao,46840,46840,2023-01-01T05:55:00,2023-12-31T19:20:00
4,australia,6,30,teste,63393,63393,2024-01-01T06:00:00,2025-08-22T18:15:00
5,australia,12,60,teste,63393,63393,2024-01-01T06:00:00,2025-08-22T18:15:00
6,brasil,6,30,treino,32275,32275,2025-01-01T06:25:00,2025-09-30T18:10:00
7,brasil,12,60,treino,31874,31874,2025-01-01T06:55:00,2025-09-30T18:10:00
8,brasil,6,30,validacao,4043,4043,2025-10-01T06:00:00,2025-10-31T18:15:00
9,brasil,12,60,validacao,4043,4043,2025-10-01T06:00:00,2025-10-31T18:15:00


## Reexecutar cálculos

Use o terminal para evitar treinos longos ocultos em uma célula:

```powershell
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py preparar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py pretreinar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py adaptar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py avaliar
```

Históricos de 30 e 60 minutos (6/12 passos). São cinco sementes, MLP direta, MLP residual e LSTM direta. Treino fonte: uma de cada 12 janelas diurnas válidas, por custo de CPU, mantendo 2013–2022; validação 2023 e teste 2024–2025 em todos os instantes elegíveis. Treino brasileiro até setembro, validação outubro, teste novembro–dezembro de 2025. Os dados são autocorrelacionados: número de instantes não equivale a observações independentes. A configuração fixa máximos de 25 épocas fonte e 40 na adaptação; são experimentos iniciais, sem busca no teste.
